# Psychological-Bias-Transfer Colab Pilot
Run these cells top-to-bottom.

In [ ]:
import os
REPO_URL = "https://github.com/coenhewes/Psychological-Bias-Transfer.git"
!if [ -d "Psychological-Bias-Transfer/.git" ]; then cd Psychological-Bias-Transfer && git fetch origin && git reset --hard origin/main; else git clone "$REPO_URL"; fi
%cd Psychological-Bias-Transfer

In [ ]:
!pip install -r requirements.txt

In [ ]:
import os
try:
    from google.colab import userdata
    _in_colab = True
except ImportError:
    userdata = None
    _in_colab = False

def _get(key):
    if _in_colab:
        try:
            return userdata.get(key)
        except Exception:
            return None
    val = os.environ.get(key)
    if val:
        return val
    env_path = ".env"
    if os.path.exists(env_path):
        for line in open(env_path):
            line = line.strip()
            if line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            if k.strip() == key:
                return v.strip().strip('"').strip("'")
    return None

HF_TOKEN = _get("HF_TOKEN")
GEMINI_API_KEY = _get("GEMINI_API_KEY") or _get("GOOGLE_API_KEY")
JUDGE_BACKEND = _get("JUDGE_BACKEND") or "gemini"
JUDGE_MODEL = _get("JUDGE_MODEL") or "gemini-3.5-flash"

os.environ["HF_TOKEN"] = HF_TOKEN or ""
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY or ""
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY or ""
os.environ["JUDGE_BACKEND"] = JUDGE_BACKEND
os.environ["JUDGE_MODEL"] = JUDGE_MODEL
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if not _in_colab:
    print("Running locally. Secrets come from os.environ or .env.")
    print("  HF_TOKEN set?", bool(HF_TOKEN))
    print("  GEMINI_API_KEY set?", bool(GEMINI_API_KEY))


In [ ]:
!python3 -c "import importlib.util; import sys; sys.exit(0 if importlib.util.find_spec('datasketch') else 1)" || pip install datasketch==1.6.5
!python3 -c "import importlib.util; import sys; sys.exit(0 if importlib.util.find_spec('sklearn') else 1)" || pip install scikit-learn>=1.6.0

In [ ]:
from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print('HF_TOKEN is not set; dataset downloads may be rate-limited.')

In [ ]:
%%bash
export HF_TOKEN JUDGE_BACKEND JUDGE_MODEL PYTORCH_CUDA_ALLOC_CONF
python3 data/corpus_builder.py \
  --source hf_csv_source \
  --hf-dataset-id solomonk/reddit_mental_health_posts \
  --hf-configs depression.csv ptsd.csv aspergers.csv adhd.csv \
  --text-fields title body \
  --treatment-subreddits depression ptsd \
  --control-candidates aspergers adhd \
  --out-dir data/processed \
  --target-tokens 200000

In [ ]:
%%bash
python3 data/corpus_validator.py \
  --treatment data/processed/treatment_corpus.jsonl \
  --control data/processed/control_corpus.jsonl \
  --out-dir data/validation

In [ ]:
%%bash
python3 training/finetune_qlora.py --model qwen2.5-7b --corpus treatment --seed 42 --config config/training_config.yaml

In [ ]:
%%bash
python3 evaluation/generate_outputs.py \
  --base-model Qwen/Qwen2.5-7B-Instruct \
  --adapter runs/qwen2.5-7b_treatment_seed42/final_adapter \
  --condition-name qwen2.5-7b_treatment_seed42 \
  --out data/generations/qwen2.5-7b_treatment_seed42.jsonl


In [ ]:
%%bash
export HF_TOKEN GEMINI_API_KEY GOOGLE_API_KEY JUDGE_BACKEND JUDGE_MODEL PYTORCH_CUDA_ALLOC_CONF
python3 evaluation/judge.py \
  --generations data/generations/qwen2.5-7b_treatment_seed42.jsonl \
  --judge "$JUDGE_BACKEND" --model "$JUDGE_MODEL" \
  --out data/judged/qwen2.5-7b_treatment_seed42.jsonl


In [ ]:
%%bash
python3 analysis/statistical_analysis.py \
  --judged-dir data/judged --out-dir results \
  --corpus-hit-rate-report data/validation/validation_report.json

In [ ]:
%%bash
python3 scripts/build_release_artifacts.py \
  --processed-dir data/processed \
  --validation-dir data/validation \
  --results-dir results \
  --out-dir data/release

## Alternative: Run on Google Cloud Vertex AI (A100 GPU)
If you want to run this training significantly faster (~1.5 hours instead of 4 hours) on a high-end NVIDIA A100 GPU, you can submit the job to Google Cloud Vertex AI using our custom script. 

**Prerequisites:**
1. Ensure you have your service account JSON key file uploaded to the notebook workspace (e.g., at `/content/citric-snow-496311.json`).
2. Ensure your `GCS_BUCKET` is configured and has been created.

In [ ]:
# Set your Vertex variables and submit the CustomJob
os.environ["GCS_BUCKET"] = "parallax-model-training-citric-snow-496311-f6"
os.environ["GCP_PROJECT"] = "citric-snow-496311-f6"
os.environ["GCP_SA_KEY"] = "/content/citric-snow-496311.json"  # Update with your actual key path

!chmod +x scripts/run_on_vertex.sh
!export GCS_BUCKET="$GCS_BUCKET" && \
 export GCP_PROJECT="$GCP_PROJECT" && \
 export GCP_SA_KEY="$GCP_SA_KEY" && \
 export HF_TOKEN="$HF_TOKEN" && \
 ./scripts/run_on_vertex.sh